# Question 1

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

In [ ]:
# --- 1. CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64  # Increased for efficiency
LEARNING_RATE = 0.001
EPOCHS = 30
NUM_CLASSES = 10

In [ ]:
# --- DATA AUGMENTATION FOR TRAINING ---
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    # 1. Flip horizontally with 50% probability
    transforms.RandomHorizontalFlip(p=0.5),
    # 2. Randomly rotate by up to 15 degrees
    transforms.RandomRotation(15),
    # 3. Randomly change brightness and contrast
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    # 4. Final conversion and normalization
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # these values are the average values over the whole ImageNet dataset
])


In [ ]:
# --- STANDARDIZED TRANSFORM FOR VALIDATION ---
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# !unzip -q /content/Dataset.zip -d /content/local_data
# print("Extracted folders:", os.listdir('/content/local_data'))

In [ ]:
# 1. Set the local root (Adjust if your zip had an extra internal folder)
# Usually, it's /content/local_data/ or /content/local_data/dataset
local_root = '/content/local_data/dataset'

# 2. Update ImageFolder paths
train_dataset = datasets.ImageFolder(
    root=os.path.join(local_root, 'train'),
    transform=train_transform  # Use the augmented transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(local_root, 'val'),
    transform=val_transform    # Use the standard transform
)

# 3. Initialize Optimized DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,    # Uses multiple CPU cores to load images
    pin_memory=True   # Speeds up data transfer to the T4 GPU
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Success! Ready to train on {len(train_dataset)} local images.")

Success! Ready to train on 3500 local images.


In [ ]:
class AssignmentCNN(nn.Module):
    def __init__(self, num_classes, verbose=False):
        super(AssignmentCNN, self).__init__()
        self.verbose = verbose

        # --- Layer 1: Conv -> MaxPool ---
        # Input: 64x64x3
        # Constraint: 3x3 kernel. Let's use padding=0 (Valid Conv) to show geometry changes.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=0)
        self.bn1 = nn.BatchNorm2d(32) # Allowed (helps training)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # --- Layer 2: Conv -> MaxPool ---
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # --- Layer 3: Conv (No Pooling) ---
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.bn3 = nn.BatchNorm2d(128)

        # --- 1 Fully Connected Layer ---
        # We must calculate the input size manually:
        # 1. Start: 64x64
        # 2. Conv1 (64-3+1) = 62x62 -> Pool1 (/2) = 31x31
        # 3. Conv2 (31-3+1) = 29x29 -> Pool2 (/2) = 14x14 (PyTorch floors 14.5)
        # 4. Conv3 (14-3+1) = 12x12
        # Final Flatten Size: 12 * 12 * 128 filters = 18,432
        self.fc = nn.Linear(12 * 12 * 128, num_classes)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # 1. Block 1
        x = self.conv1(x)
        if self.verbose: print(f"After Conv1: {x.shape}")
        x = self.pool1(F.relu(self.bn1(x)))
        if self.verbose: print(f"After Pool1: {x.shape}")

        # 2. Block 2
        x = self.conv2(x)
        if self.verbose: print(f"After Conv2: {x.shape}")
        x = self.pool2(F.relu(self.bn2(x)))
        if self.verbose: print(f"After Pool2: {x.shape}")

        # 3. Block 3
        x = self.conv3(x)
        if self.verbose: print(f"After Conv3: {x.shape}")
        x = F.relu(self.bn3(x))

        # 4. Flatten & FC
        x = torch.flatten(x, 1)
        if self.verbose: print(f"After Flatten: {x.shape}")

        x = self.dropout(x)
        x = self.fc(x)
        if self.verbose: print(f"Final Output: {x.shape}")

        return x

In [ ]:
# Initialize with verbose=True to print shapes
print("--- Verifying Architecture Dimensions ---")
temp_model = AssignmentCNN(num_classes=10, verbose=True).to(DEVICE)
dummy_input = torch.randn(1, 3, 64, 64).to(DEVICE)
_ = temp_model(dummy_input)
print("-----------------------------------------")

--- Verifying Architecture Dimensions ---
After Conv1: torch.Size([1, 32, 62, 62])
After Pool1: torch.Size([1, 32, 31, 31])
After Conv2: torch.Size([1, 64, 29, 29])
After Pool2: torch.Size([1, 64, 14, 14])
After Conv3: torch.Size([1, 128, 12, 12])
After Flatten: torch.Size([1, 18432])
Final Output: torch.Size([1, 10])
-----------------------------------------


In [ ]:
# --- 4. TRAINING & VALIDATION ENGINE ---
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    if phase == 'train':
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # Disable gradient tracking if validating
    with torch.set_grad_enabled(phase == 'train'):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            if phase == 'train':
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if phase == 'train':
                loss.backward()
                optimizer.step()

            # Stats
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
# --- 5. EXECUTION ---
if __name__ == "__main__":
    model = AssignmentCNN(NUM_CLASSES, verbose=False).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    print(f"Training on: {DEVICE}")
    print("-" * 30)

    # Example Training Loop (uncomment when data is ready)
    for epoch in range(EPOCHS):
      train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, phase='train')
      val_loss, val_acc = run_epoch(model, val_loader, criterion, phase='val')

      # Step the scheduler based on validation loss
      scheduler.step(val_loss)

      print(f"Epoch {epoch+1}: Train Acc {train_acc:.2f}% | Val Acc {val_acc:.2f}%")

    # Verification Step
    dummy_input = torch.randn(1, 3, 64, 64).to(DEVICE)
    print("Architecture verified. Output shape:", model(dummy_input).shape)

Training on: cuda
------------------------------
Epoch 1: Train Acc 37.00% | Val Acc 46.80%
Epoch 2: Train Acc 45.14% | Val Acc 46.00%
Epoch 3: Train Acc 49.23% | Val Acc 48.20%
Epoch 4: Train Acc 53.66% | Val Acc 55.20%
Epoch 5: Train Acc 56.40% | Val Acc 51.00%
Epoch 6: Train Acc 57.17% | Val Acc 56.00%
Epoch 7: Train Acc 59.86% | Val Acc 54.00%
Epoch 8: Train Acc 62.54% | Val Acc 51.60%
Epoch 9: Train Acc 63.54% | Val Acc 57.00%
Epoch 10: Train Acc 64.20% | Val Acc 56.40%
Epoch 11: Train Acc 66.46% | Val Acc 59.20%
Epoch 12: Train Acc 67.97% | Val Acc 57.00%
Epoch 13: Train Acc 67.46% | Val Acc 58.40%
Epoch 14: Train Acc 69.54% | Val Acc 55.40%
Epoch 15: Train Acc 69.37% | Val Acc 57.80%
Epoch 16: Train Acc 71.06% | Val Acc 60.80%
Epoch 17: Train Acc 71.66% | Val Acc 62.00%
Epoch 18: Train Acc 71.49% | Val Acc 56.40%
Epoch 19: Train Acc 72.51% | Val Acc 61.40%
Epoch 20: Train Acc 74.29% | Val Acc 63.20%
Epoch 21: Train Acc 74.94% | Val Acc 63.20%
Epoch 22: Train Acc 77.00% | Val Acc

In [ ]:
# 1. Define the Test Loader (No Augmentation, just normalization)
test_dataset = datasets.ImageFolder(
    root=os.path.join(local_root, 'test'), # Points to /content/local_data/test
    transform=val_transform  # Vital: Use the 'clean' transform, not the training one
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Test Set Loaded: {len(test_dataset)} images.")

def evaluate_model(model, loader):
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0

    # We don't need gradients for testing
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    final_acc = 100 * correct / total
    return final_acc

# Run the evaluation
print("Running Final Test Evaluation...")
test_accuracy = evaluate_model(model, test_loader)
print(f"Final Test Accuracy: {test_accuracy:.2f}%")

Test Set Loaded: 1000 images.
Running Final Test Evaluation...
Final Test Accuracy: 66.10%
